# Credence: Multi-Model EQLR Benchmark

**Research question:** Is Epistemic Qualifier Loss (EQL) model-agnostic, or specific to the model families we already tested?

**Prior results:**
| Model | Family | n (unguarded) | EQLR |
|---|---|---|---|
| Claude Haiku | Anthropic | 50 | 46% [32–61%] |
| Qwen-2.5-1.5B | Alibaba | 154 | 50% [42–58%] |

**This notebook adds:**
| Model | Family | Size | Notes |
|---|---|---|---|
| Qwen-2.5-7B-Instruct | Alibaba | 7B | open, no token |
| Mistral-7B-Instruct-v0.3 | Mistral AI | 7B | open, no token |
| Phi-3.5-mini-Instruct | Microsoft | 3.8B | open, no token |
| Llama-3.2-3B-Instruct | Meta | 3B | requires HF_TOKEN + Meta license |
| Gemma-2-9B-it | Google | 9B | requires HF_TOKEN + Google license |
| Llama-3.1-8B-Instruct | Meta | 8B | requires HF_TOKEN + Meta license |

**Metric:** EQLR — does the compressed output still contain any uncertainty qualifier?  
Measured directly on compressor output. No downstream model needed.

**Expected result:** ~40–60% EQLR unguarded, 0% probe-blocked — across all model families.

---

> **Gated models (Llama-3.2-3B, Gemma-2-9B, Llama-3.1-8B):** add `HF_TOKEN` as a Kaggle Secret and accept each license on HuggingFace before running.  
> Without a token, all three are automatically skipped — the three open models still run.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'transformers', 'accelerate', 'bitsandbytes', '-q'], check=True)
print('Dependencies ready')

In [ ]:
import json, re, time, os, gc, random, collections
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    TOTAL_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {TOTAL_VRAM_GB:.1f}GB')
else:
    TOTAL_VRAM_GB = 0

# ── HuggingFace token (set via Kaggle Secrets → HF_TOKEN) ───────────────────
HF_TOKEN = os.environ.get('HF_TOKEN', None)

if HF_TOKEN:
    try:
        from huggingface_hub import login as hf_login
        hf_login(token=HF_TOKEN, add_to_git_credential=False)
        print('HF_TOKEN found — logged in to HuggingFace. Gated models will be attempted.')
    except Exception as e:
        print(f'HF login warning: {e}')
else:
    print('No HF_TOKEN — gated models (Llama-3.2-3B, Gemma-2-9B, Llama-3.1-8B) will be skipped.')
    print('Add HF_TOKEN as a Kaggle Secret to enable them.')

# Gated models are skipped automatically when no token is present
SKIP_GATED = not bool(HF_TOKEN)


## Model Registry

In [ ]:
# Model registry
# use_4bit=True  → BitsAndBytes 4-bit quantization (for 7B+ on T4 15.6GB VRAM)
# use_4bit=False → fp16 (for models ≤4B that fit directly)
# hf_gated=True  → requires HF_TOKEN + license acceptance on HuggingFace
# torch_dtype    → override dtype (Gemma-2 prefers bfloat16)

MODELS = [
    {
        'id':         'Qwen/Qwen2.5-7B-Instruct',
        'family':     'Qwen',
        'size':       '7B',
        'use_4bit':   True,
        'hf_gated':   False,
        'torch_dtype': 'float16',
        'license_url': None,
    },
    {
        'id':         'mistralai/Mistral-7B-Instruct-v0.3',
        'family':     'Mistral',
        'size':       '7B',
        'use_4bit':   True,
        'hf_gated':   False,
        'torch_dtype': 'float16',
        'license_url': None,
    },
    {
        'id':         'microsoft/Phi-3.5-mini-instruct',
        'family':     'Phi',
        'size':       '3.8B',
        'use_4bit':   False,
        'hf_gated':   False,
        'torch_dtype': 'float16',
        'license_url': None,
    },
    {
        'id':         'meta-llama/Llama-3.2-3B-Instruct',
        'family':     'Llama',
        'size':       '3B',
        'use_4bit':   False,   # 3B fits in fp16 (~6GB)
        'hf_gated':   True,    # accept at: huggingface.co/meta-llama/Llama-3.2-3B-Instruct
        'torch_dtype': 'float16',
        'license_url': 'https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct',
    },
    {
        'id':         'google/gemma-2-9b-it',
        'family':     'Gemma',
        'size':       '9B',
        'use_4bit':   True,    # 9B → 4-bit to fit in T4 VRAM
        'hf_gated':   True,    # accept at: huggingface.co/google/gemma-2-9b-it
        'torch_dtype': 'bfloat16',  # Gemma-2 trained in bfloat16
        'license_url': 'https://huggingface.co/google/gemma-2-9b-it',
    },
    {
        'id':         'meta-llama/Llama-3.1-8B-Instruct',
        'family':     'Llama',
        'size':       '8B',
        'use_4bit':   True,
        'hf_gated':   True,    # accept at: huggingface.co/meta-llama/Llama-3.1-8B-Instruct
        'torch_dtype': 'float16',
        'license_url': 'https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct',
    },
]

# Build active list
ACTIVE_MODELS = []
for m in MODELS:
    if m['hf_gated'] and SKIP_GATED:
        url = m.get('license_url', '')
        print(f"  ✗ Skipping {m['id']} (gated, no HF_TOKEN)")
        if url:
            print(f"    → Accept license at: {url}")
    else:
        ACTIVE_MODELS.append(m)
        flag   = '(4-bit)' if m['use_4bit'] else '(fp16)' 
        gated  = '⚠ gated' if m['hf_gated'] else 'open'
        dtype  = m.get('torch_dtype', 'float16')
        print(f"  ✓ {m['id']} [{m['family']}, {m['size']}] {flag} dtype={dtype} {gated}")

print(f'\n{len(ACTIVE_MODELS)} / {len(MODELS)} models active.')


## Load EQL-Bench v2

In [ ]:
_DATASET_SLUG = 'chakradharvijayarao/eql-bench-v2-epistemic-qualifier-loss-benchmark'
_DOWNLOAD_DIR = '/kaggle/working/eql_data'

def _print_kaggle_inputs():
    base = '/kaggle/input'
    if not os.path.exists(base): return
    print('DEBUG /kaggle/input contents:')
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, '').count(os.sep)
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(root)}/')
        for fn in files:
            print(f'{indent}  {fn}')

def _download_if_missing():
    dest_file = os.path.join(_DOWNLOAD_DIR, 'eql_bench_v2.json')
    if os.path.exists(dest_file): return
    print(f'Dataset not mounted — downloading {_DATASET_SLUG} ...')
    os.makedirs(_DOWNLOAD_DIR, exist_ok=True)
    kaggle_bin = subprocess.run(['which', 'kaggle'], capture_output=True, text=True).stdout.strip() or 'kaggle'
    result = subprocess.run(
        [kaggle_bin, 'datasets', 'download', _DATASET_SLUG, '--unzip', '-p', _DOWNLOAD_DIR],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        print(f'  Downloaded to {_DOWNLOAD_DIR}')
        for root, dirs, files in os.walk(_DOWNLOAD_DIR):
            for fn in files:
                src = os.path.join(root, fn)
                dst = os.path.join(_DOWNLOAD_DIR, fn)
                if src != dst: os.rename(src, dst)
    else:
        print(f'  Download failed: {result.stderr.strip()[:200]}')

_DATA_CANDIDATES = [
    os.path.join(_DOWNLOAD_DIR, 'eql_bench_v2.json'),
    '/kaggle/input/eql-bench-v2-epistemic-qualifier-loss-benchmark/eql_bench_v2.json',
    '/kaggle/input/eql-bench-v2-epistemic-qualifier-loss-benchmark/credence-eql-bench/eql_bench_v2.json',
    '/kaggle/input/credence-eql-bench/eql_bench_v2.json',
]

if os.path.exists('/kaggle'):
    _print_kaggle_inputs()
    _download_if_missing()

ALL_SCENARIOS = None
for _p in _DATA_CANDIDATES:
    if os.path.exists(_p):
        with open(_p) as _f:
            _raw = json.load(_f)
        ALL_SCENARIOS = _raw.get('scenarios', _raw) if isinstance(_raw, dict) else _raw
        print(f'Loaded {len(ALL_SCENARIOS)} scenarios from {_p}')
        break

if ALL_SCENARIOS is None:
    raise RuntimeError('Dataset not found at any candidate path.')

# Use all 280 explicit scenarios — maximises unguarded n for reliable CI
EXPLICIT = [s for s in ALL_SCENARIOS if not s.get('is_ghost', False)]
GHOST    = [s for s in ALL_SCENARIOS if s.get('is_ghost', False)]
print(f'Explicit: {len(EXPLICIT)}  Ghost: {len(GHOST)}')
print(f'Running all {len(EXPLICIT)} explicit scenarios per model (not a sample).')

## Faithfulness Probe (228 markers)

In [ ]:
_PROBE_MARKERS = frozenset({
    'not certain','not sure','uncertain','tentative','unverified',
    'approximately','roughly','i think','i believe',"i'm not",
    'might be','might not','may be','possibly','perhaps',
    "i'd verify",'need to check','should verify','to verify','approx','tbd',
    'probably','maybe','provisionally','preliminary','supposedly',
    'ambiguous','unclear',"hasn't clarified",'not yet clarified',
    'unconfirmed','not confirmed','not yet confirmed','open question','still open',
    'needs verification','need to verify','not yet decided','not decided',
    'to be determined','to be confirmed',"haven't confirmed","haven't verified",
    "haven't checked",'depending on','depends on whether','subject to',
    'contingent on','once we confirm','once we verify','pending confirmation',
    'as far as i know','to my knowledge','to my understanding',
    'if i recall','i seem to recall','last time i checked','best of my knowledge',
    'working theory','my assumption',"i'm assuming",'in theory',
    'could be wrong','not 100%','not entirely sure',
    'the vendor said','they mentioned','reportedly','supposedly',
    'the docs say','i read somewhere','heard that','we were told',
    'give or take','ballpark','order of magnitude','in the range of',
    'somewhere around','plus or minus','estimated at',
    'untested','not yet tested',"haven't tested",'not benchmarked',
    'iirc','afaik','if i recall correctly','from memory','off the top of my head',
    "i'm unsure",'unsure','not sure which','unsure of',
    'vendor claims','sales rep said','they told us','per the vendor',
    'from the vendor','vendor estimate','vendor said','vendor mentioned',
    'sales call','the demo showed','account rep said','sales team said',
    'from a quote','per the quote','based on a quote','their estimate',
    'from a blog post','i read online','saw online','from a forum',
    'per a forum post','from a slack message','in a slack',
    'from a tweet','someone mentioned','i heard',
    'from a stackoverflow','from stackoverflow','per stackoverflow',
    'from a reddit post','per reddit',
    'not production-tested','not load-tested','never tested in production',
    'works in theory','not stress-tested',
    'assuming that',"if that's right","if that's correct",
    "wasn't sure","weren't sure","wasn't certain","weren't certain",
    "hadn't verified","hadn't confirmed","hadn't checked",
    "didn't confirm","didn't verify","didn't check",
    'might have been','may have been','could have been',
    'might ','may have',
    "haven't seen","hasn't finalized",
    'pending verification','requires verification',
    'not fully confirmed','not been confirmed','not yet finalized',
    'cannot definitively','not definitively','unresolved','still pending',
    'needs confirmation',
    # EQL-Bench v2 expansion
    'seems to','seem to','appears to','appear to','seems like',
    'colleague said','a colleague','colleague mentioned',
    'sales claimed','sales said','inferred from','logs show',
    'not documented','undocumented','no official','no official documentation',
    'we estimate','we think','we suspect','i suspect',
    'not verified','not reproduced',"haven't measured",
    'back-of-envelope','back of envelope','rough estimate',
    'blog post','a preprint','not peer-reviewed',
    'informal guidance','informal advice','extrapolated from',
    'not tested in','not tested on','marketing materials',
    'consultant suggested','conflicting reports','conflicting data',
    'may not apply','may not hold','forum post','forum thread',
    'rough reading','vendor says','vendor claims',
    "vendor's guide","vendor's whitepaper","vendor's documentation",
    'early exploration','early discussion','nothing finalized','nothing decided',
    'nothing ratified',"hasn't decided","haven't decided",
    'is preliminarily','preliminarily','not formally',
    'a case report','phase 1 safety','single case','no control group',
    'not yet assessed','not announced','sales deck',
    'a paper claimed','a paper claims','status page shows',
    'someone said','someone told','slack thread',
})

# Broader hedge words for EQLR scoring (catches paraphrased hedges)
_EQLR_HEDGES = [
    'could', 'might', 'may ', 'likely', 'unlikely', 'possibly', 'uncertain',
    'unclear', 'perhaps', 'potentially', 'approximately', 'roughly',
    'probably', 'not sure', 'unsure', 'unconfirmed', 'seems', 'appear',
    'tentatively', 'reportedly', 'allegedly', 'supposedly', 'estimated',
]

def probe_fires(text: str) -> bool:
    lower = text.lower()
    return any(m in lower for m in _PROBE_MARKERS)

def score_eqlr(compressed: str) -> bool:
    """True = EQL event (qualifier stripped). False = qualifier survived."""
    lower = compressed.lower()
    if any(m in lower for m in _PROBE_MARKERS): return False
    if any(h in lower for h in _EQLR_HEDGES):  return False
    return True  # no qualifier found

n_blocked   = sum(1 for s in EXPLICIT if probe_fires(s['uncertain_statement']))
n_unguarded = len(EXPLICIT) - n_blocked
print(f'Probe markers: {len(_PROBE_MARKERS)}')
print(f'Probe coverage on EXPLICIT: {n_blocked}/{len(EXPLICIT)} ({100*n_blocked/len(EXPLICIT):.1f}%)')
print(f'Unguarded scenarios (will be compressed): {n_unguarded}')

## Helper Functions

In [ ]:
_COMPRESS_PROMPT = (
    'You are a context compression assistant. '
    'Summarize the following message concisely in 1-2 sentences, keeping the key technical fact.\n\n'
    'Message: {text}\n\n'
    'Summary:'
)

def load_model(model_cfg: dict):
    model_id   = model_cfg['id']
    use_4bit   = model_cfg['use_4bit']
    hf_gated   = model_cfg.get('hf_gated', False)
    dtype_str  = model_cfg.get('torch_dtype', 'float16')
    token_arg  = HF_TOKEN if hf_gated else None

    dtype_map  = {'float16': torch.float16, 'bfloat16': torch.bfloat16, 'float32': torch.float32}
    torch_dtype = dtype_map.get(dtype_str, torch.float16)

    print(f'\nLoading {model_id} [{"4-bit" if use_4bit else dtype_str}]...')
    t0 = time.time()

    tok = AutoTokenizer.from_pretrained(
        model_id, trust_remote_code=True, token=token_arg
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    if use_4bit:
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch_dtype,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
        )
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_cfg,
            device_map='auto',
            trust_remote_code=True,
            token=token_arg,
        )
    else:
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch_dtype,
            device_map='auto',
            trust_remote_code=True,
            token=token_arg,
        )

    mdl.eval()
    elapsed = time.time() - t0
    if torch.cuda.is_available():
        vram_used = torch.cuda.memory_allocated() / 1e9
        print(f'Loaded in {elapsed:.0f}s  VRAM: {vram_used:.1f}/{TOTAL_VRAM_GB:.1f}GB')
    else:
        print(f'Loaded in {elapsed:.0f}s (CPU)')
    return tok, mdl


def unload_model(mdl):
    del mdl
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        vram_used = torch.cuda.memory_allocated() / 1e9
        print(f'Unloaded. VRAM freed → {vram_used:.1f}GB remaining')


@torch.inference_mode()
def compress(tok, mdl, text: str, max_new_tokens: int = 80) -> str:
    prompt   = _COMPRESS_PROMPT.format(text=text)
    messages = [{'role': 'user', 'content': prompt}]
    try:
        inp_text = tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        inp_text = f'<|user|>\n{prompt}\n<|assistant|>\n'

    inputs  = tok(inp_text, return_tensors='pt', truncation=True, max_length=512).to(mdl.device)
    out_ids = mdl.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        repetition_penalty=1.3,
        pad_token_id=tok.eos_token_id,
    )
    new_ids = out_ids[0, inputs['input_ids'].shape[1]:]
    return tok.decode(new_ids, skip_special_tokens=True).strip()


def bootstrap_ci(values, n_boot=2000, ci=0.95, seed=42):
    n = len(values)
    if n == 0: return (0.0, 0.0)
    rng = random.Random(seed)
    boots = sorted(
        sum(values[rng.randint(0, n-1)] for _ in range(n)) / n
        for _ in range(n_boot)
    )
    lo = int((1 - ci) / 2 * n_boot)
    hi = int((1 - (1 - ci) / 2) * n_boot)
    return (round(boots[lo], 4), round(boots[hi], 4))


print('Helpers defined.')


## Main Loop — Run Each Model

In [ ]:
# Results stored per model
all_model_results = {}

for model_cfg in ACTIVE_MODELS:
    model_id = model_cfg['id']
    print(f'\n{"="*70}')
    print(f'MODEL: {model_id}  [{model_cfg["family"]}, {model_cfg["size"]}]')
    print(f'{"="*70}')

    # ── Load ──────────────────────────────────────────────────────────────
    try:
        tok, mdl = load_model(model_cfg)
    except Exception as e:
        print(f'FAILED to load {model_id}: {e}')
        all_model_results[model_id] = {'error': str(e)}
        continue

    # ── Smoke test ────────────────────────────────────────────────────────
    _test = "Auth tokens appear to expire after roughly 1 hour — inferred from logs, haven't confirmed."
    _out  = compress(tok, mdl, _test)
    print(f'Smoke test input:  {_test}')
    print(f'Smoke test output: {_out}')
    print(f'EQL event: {score_eqlr(_out)}')

    # ── Compress all 280 explicit scenarios ───────────────────────────────
    scenario_results = []
    t_start = time.time()
    n_total = len(EXPLICIT)

    for i, scenario in enumerate(EXPLICIT):
        stmt    = scenario['uncertain_statement']
        blocked = probe_fires(stmt)

        if blocked:
            compressed = stmt        # probe keeps original
            eql_event  = False       # can't be EQL — original has the marker
        else:
            compressed = compress(tok, mdl, stmt)
            eql_event  = score_eqlr(compressed)

        scenario_results.append({
            'scenario_id':    scenario['scenario_id'],
            'domain':         scenario['domain'],
            'qualifier_type': scenario['qualifier_type'],
            'probe_blocked':  blocked,
            'compressed':     compressed,
            'eql_event':      eql_event,
        })

        if (i + 1) % 40 == 0 or (i + 1) == n_total:
            elapsed   = time.time() - t_start
            rate      = (i + 1) / max(elapsed, 0.001)
            eta       = (n_total - i - 1) / rate
            done      = scenario_results
            unguarded = [r for r in done if not r['probe_blocked']]
            cur_eqlr  = (sum(r['eql_event'] for r in unguarded) / len(unguarded)) if unguarded else 0.0
            print(f'  [{i+1:3d}/{n_total}] {elapsed:4.0f}s  ETA {eta:4.0f}s  unguarded_EQLR={cur_eqlr:.1%}')

    elapsed_total = time.time() - t_start

    # ── Per-model summary ─────────────────────────────────────────────────
    blocked_list   = [r for r in scenario_results if r['probe_blocked']]
    unguarded_list = [r for r in scenario_results if not r['probe_blocked']]
    eql_vals       = [1.0 if r['eql_event'] else 0.0 for r in unguarded_list]

    unguarded_eqlr = sum(eql_vals) / len(eql_vals) if eql_vals else 0.0
    eqlr_ci        = bootstrap_ci(eql_vals) if eql_vals else (0.0, 0.0)

    # Domain breakdown
    domain_eqlr = {}
    for dom in sorted({r['domain'] for r in unguarded_list}):
        sub = [1.0 if r['eql_event'] else 0.0 for r in unguarded_list if r['domain'] == dom]
        domain_eqlr[dom] = {
            'n': len(sub), 'eqlr': round(sum(sub)/len(sub), 4) if sub else 0.0
        }

    # Qualifier type breakdown
    qtype_eqlr = {}
    for qt in sorted({r['qualifier_type'] for r in unguarded_list}):
        sub = [1.0 if r['eql_event'] else 0.0 for r in unguarded_list if r['qualifier_type'] == qt]
        qtype_eqlr[qt] = {
            'n': len(sub), 'eqlr': round(sum(sub)/len(sub), 4) if sub else 0.0
        }

    model_summary = {
        'model_id':        model_id,
        'family':          model_cfg['family'],
        'size':            model_cfg['size'],
        'use_4bit':        model_cfg['use_4bit'],
        'n_total':         len(scenario_results),
        'n_blocked':       len(blocked_list),
        'n_unguarded':     len(unguarded_list),
        'probe_block_rate': round(len(blocked_list)/len(scenario_results), 4),
        'unguarded_eqlr':  round(unguarded_eqlr, 4),
        'eqlr_ci95':       eqlr_ci,
        'elapsed_s':       round(elapsed_total, 1),
        'by_domain':       domain_eqlr,
        'by_qualifier_type': qtype_eqlr,
    }

    print(f'\n  ── {model_id} summary ──')
    print(f'  n_total={model_summary["n_total"]}  n_blocked={model_summary["n_blocked"]}  n_unguarded={model_summary["n_unguarded"]}')
    print(f'  Unguarded EQLR: {unguarded_eqlr:.1%}  95%CI [{eqlr_ci[0]:.1%}, {eqlr_ci[1]:.1%}]')
    print(f'  Probe-blocked EQLR: 0.0% (deterministic)')

    all_model_results[model_id] = {
        'summary':  model_summary,
        'scenarios': scenario_results,
    }

    # ── Unload ────────────────────────────────────────────────────────────
    unload_model(mdl)
    del tok
    gc.collect()

    # ── Save checkpoint after each model (crash-safe) ─────────────────────
    ckpt_path = f'/kaggle/working/checkpoint_{model_cfg["family"].lower()}.json'
    with open(ckpt_path, 'w') as f:
        json.dump({'summary': model_summary, 'scenarios': scenario_results}, f, indent=2)
    print(f'  Checkpoint saved: {ckpt_path}')

print(f'\n{"="*70}')
print(f'All {len(all_model_results)} models complete.')

## Results — Comparison Table

In [ ]:
print('=' * 76)
print('CREDENCE MULTI-MODEL EQLR BENCHMARK')
print('Metric: EQLR = fraction of unguarded compressions that strip the qualifier')
print('Probe-blocked EQLR = 0% by construction (original preserved)')
print('=' * 76)
print()
print(f'  {"Model":<40} {"Family":<10} {"Size":<6} {"n_ung":>6}  {"EQLR":>8}  {"95% CI":>18}')
print(f'  {"-"*40} {"-"*10} {"-"*6} {"-"*6}  {"-"*8}  {"-"*18}')

# Prior results for reference
PRIOR = [
    ('Claude Haiku',       'Anthropic', 'small', 50,  0.460, (0.318, 0.607)),
    ('Qwen-2.5-1.5B',     'Alibaba',   '1.5B',  154, 0.500, (0.422, 0.578)),
]
for name, fam, sz, n_u, eqlr, ci in PRIOR:
    ci_str = f'[{ci[0]:.1%}, {ci[1]:.1%}]'
    print(f'  {name:<40} {fam:<10} {sz:<6} {n_u:>6}  {eqlr:>7.1%}  {ci_str:>18}  (prior)')

print(f'  {"-"*40} {"-"*10} {"-"*6} {"-"*6}  {"-"*8}  {"-"*18}')

# This run
for model_id, data in all_model_results.items():
    if 'error' in data:
        print(f'  {model_id:<40}  ERROR: {data["error"][:40]}')
        continue
    s      = data['summary']
    ci_str = f'[{s["eqlr_ci95"][0]:.1%}, {s["eqlr_ci95"][1]:.1%}]'
    print(f'  {s["model_id"]:<40} {s["family"]:<10} {s["size"]:<6} {s["n_unguarded"]:>6}  {s["unguarded_eqlr"]:>7.1%}  {ci_str:>18}  ← NEW')

print()
print('  Probe-blocked EQLR = 0.0% for ALL models (deterministic — probe fires → original preserved)')
print()

# Domain breakdown across models
print('EQLR BY DOMAIN (unguarded, this run):')
print(f'  {"Domain":<16}', end='')
active_ids = [m['id'] for m in ACTIVE_MODELS if m['id'] in all_model_results and 'error' not in all_model_results[m['id']]]
for mid in active_ids:
    fam = all_model_results[mid]['summary']['family']
    print(f'  {fam:<10}', end='')
print()
all_domains = sorted({dom for mid in active_ids for dom in all_model_results[mid]['summary']['by_domain']})
for dom in all_domains:
    print(f'  {dom:<16}', end='')
    for mid in active_ids:
        d = all_model_results[mid]['summary']['by_domain'].get(dom, {})
        eqlr = d.get('eqlr', None)
        val  = f'{eqlr:.1%}' if eqlr is not None else '—'
        print(f'  {val:<10}', end='')
    print()

print()
print('EQLR BY QUALIFIER TYPE (unguarded, this run):')
print(f'  {"Qualifier":<22}', end='')
for mid in active_ids:
    fam = all_model_results[mid]['summary']['family']
    print(f'  {fam:<10}', end='')
print()
all_qtypes = sorted({qt for mid in active_ids for qt in all_model_results[mid]['summary']['by_qualifier_type']})
for qt in all_qtypes:
    print(f'  {qt:<22}', end='')
    for mid in active_ids:
        d = all_model_results[mid]['summary']['by_qualifier_type'].get(qt, {})
        eqlr = d.get('eqlr', None)
        val  = f'{eqlr:.1%}' if eqlr is not None else '—'
        print(f'  {val:<10}', end='')
    print()

print(f'\n{"="*76}')

In [ ]:
# Save full results
out_path = ('/kaggle/working/multimodel_eqlr_results.json'
            if os.path.exists('/kaggle/working') else 'multimodel_eqlr_results.json')

output = {
    'benchmark':     'EQL-Bench v2',
    'n_scenarios':   len(EXPLICIT),
    'probe_markers': len(_PROBE_MARKERS),
    'prior_results': [
        {'model': 'Claude Haiku',   'family': 'Anthropic', 'size': 'small',
         'n_unguarded': 50,  'unguarded_eqlr': 0.460, 'eqlr_ci95': [0.318, 0.607]},
        {'model': 'Qwen-2.5-1.5B', 'family': 'Alibaba',   'size': '1.5B',
         'n_unguarded': 154, 'unguarded_eqlr': 0.500, 'eqlr_ci95': [0.422, 0.578]},
    ],
    'models': {
        mid: {
            'summary':   data['summary'],
            'scenarios': data['scenarios'],
        } if 'error' not in data else {'error': data['error']}
        for mid, data in all_model_results.items()
    },
}

with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'Full results saved to {out_path}')
print(f'File size: {os.path.getsize(out_path)/1e6:.1f}MB')

## Interpretation

### What a consistent result looks like

If all models cluster at **~40–60% unguarded EQLR** and **0% probe-blocked EQLR**, the model-agnostic claim is confirmed across:
- 3 size classes: 3B, 3.8B, 7–9B
- 5 model families: Qwen, Mistral, Phi, Llama, Gemma
- 3 organisations: Alibaba, Mistral AI, Microsoft, Meta, Google
- Combined with prior results: 6 families including Anthropic Haiku

### Why the range isn't exactly the same

EQLR variance comes from two sources:
1. **Model verbosity** — models producing longer summaries are more likely to include hedging words by chance
2. **Instruction following** — models that follow the compression prompt more aggressively produce tighter summaries that drop qualifiers more often

The key claim is directional: **unguarded EQLR >> 0%, probe-blocked EQLR = 0%** for every model tested.

### Gemma-2 note

Gemma-2-9B uses `bfloat16` compute dtype (its native training dtype) even in 4-bit mode. This prevents dtype mismatch warnings and gives the most accurate results for that model.

### Full evidence table (after run)

| Model | Family | Size | n (unguarded) | EQLR | 95% CI |
|---|---|---|---|---|---|
| Claude Haiku | Anthropic | small | 50 | 46% | [32%, 61%] |
| Qwen-2.5-1.5B | Alibaba | 1.5B | 154 | 50% | [42%, 58%] |
| Qwen-2.5-7B | Alibaba | 7B | TBD | TBD | TBD |
| Mistral-7B | Mistral AI | 7B | TBD | TBD | TBD |
| Phi-3.5-mini | Microsoft | 3.8B | TBD | TBD | TBD |
| Llama-3.2-3B | Meta | 3B | TBD | TBD | TBD |
| Gemma-2-9B | Google | 9B | TBD | TBD | TBD |
| Llama-3.1-8B | Meta | 8B | TBD | TBD | TBD |
